# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SymbolPamnani/Flyrank-ML-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Answer

### Finding 1 — Content lifecycle

The FlyRank report compares pages that are gaining traffic with pages that are losing traffic. It reports that growing pages were younger on average than declining pages: about 185 days versus 228 days. Word count was nearly the same between the two groups.

For this finding, the decline/growth label comes from **Trend Direction**, which is based on the last 30 days compared with the 30 days before that. The report defines `Down` as more than 10% decline and `Up` as more than 10% growth.

The comparison supports an **observed association** between content age and the current trend groups. However, the validation design is a descriptive cohort comparison, not a client-held-out or time-forward predictive validation. It therefore does not establish that increasing or reducing content age will cause the observed change. A stronger validation would test the relationship on unseen clients or a later time period.

### Finding 2 — Freshness

The report also compares growth-to-decline ratios across freshness windows. The 31–90 day freshness group has a reported growth-to-decline ratio of about 5.43:1, while the 181–360 day group is about 3.02:1. The report separately compares older pages that were recently refreshed with older pages that had not been refreshed recently.

Again, the trend grouping is based on the observed traffic-change label rather than a future intervention outcome. The evidence is useful for describing patterns in the dataset, but the comparison itself does not provide a randomized or time-forward test of whether refreshing a page caused the subsequent difference.

My methodology question is therefore: **Would the freshness relationship remain when pages are evaluated on a future outcome window and the validation groups or holds out clients?** That would make the evidence stronger for decision-support use.


In [1]:
# Section 1 — reproduce the label definition and show the basic label counts

import pandas as pd

DATA_PATH = "content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

# The model's decline-proxy label
y = (df["trend_direction"] == "down").astype(int)

print("Dataset shape:", df.shape)

print("\nDecline-proxy label:")
print("0 = not down")
print("1 = down")

print("\nLabel counts:")
print(y.value_counts().sort_index())

print("\nDecline-proxy rate:", f"{y.mean():.3f}")
print("Decline-proxy rate (%):", f"{y.mean() * 100:.1f}%")

print("\nTrend direction counts:")
print(df["trend_direction"].value_counts(dropna=False))

# Check that the label is actually based on trend_direction.
assert set(y.unique()).issubset({0, 1})
assert len(y) == len(df)

print("\nLabel check passed.")

Dataset shape: (30000, 44)

Decline-proxy label:
0 = not down
1 = down

Label counts:
trend_direction
0    13738
1    16262
Name: count, dtype: int64

Decline-proxy rate: 0.542
Decline-proxy rate (%): 54.2%

Trend direction counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Label check passed.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## Answer

I compare the same Week-5 Logistic Regression model under two validation designs.

The row-random split is shown as the **before** comparison because rows from the same client can appear in both training and test data. This can make the evaluation optimistic when pages from one client share characteristics.

The **after** evaluation uses a client-grouped split. All rows from a client stay in either training or test, so the test set contains clients that were not used for training.

The primary metric is Precision@50 because the practical use is to prioritize a small review queue. I also print the test base rate beside the metric.

The key audit question is whether the measured Precision@50 changes when the evaluation is made client-disjoint.


In [2]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupShuffleSplit

# ---------------------------------------------------------
# 1. Load data and define target/features
# ---------------------------------------------------------

DATA_PATH = "content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

target = (df["trend_direction"] == "down").astype(int)

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
]

feature_columns = numeric_features + categorical_features

X = df[feature_columns]
groups = df["client_id"]

# ---------------------------------------------------------
# 2. Same Week-5 model
# ---------------------------------------------------------

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median", add_indicator=True)
        ),
        (
            "scaler",
            StandardScaler()
        ),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ]
)

def make_model():
    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "classifier",
                LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    random_state=42
                )
            ),
        ]
    )

def precision_at_k(scores, labels, k=50):
    scores = np.asarray(scores)
    labels = np.asarray(labels)

    top_k = np.argsort(-scores)[:k]

    return labels[top_k].mean()

# ---------------------------------------------------------
# 3. BEFORE — ordinary row-random split
# ---------------------------------------------------------

X_train_random, X_test_random, y_train_random, y_test_random = (
    train_test_split(
        X,
        target,
        test_size=0.20,
        random_state=42,
        stratify=target
    )
)

random_model = make_model()

random_model.fit(
    X_train_random,
    y_train_random
)

random_scores = random_model.predict_proba(
    X_test_random
)[:, 1]

random_p50 = precision_at_k(
    random_scores,
    y_test_random,
    k=50
)

# ---------------------------------------------------------
# 4. AFTER — client-grouped split
# ---------------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        X,
        target,
        groups=groups
    )
)

X_train_grouped = X.iloc[train_idx]
X_test_grouped = X.iloc[test_idx]

y_train_grouped = target.iloc[train_idx]
y_test_grouped = target.iloc[test_idx]

grouped_model = make_model()

grouped_model.fit(
    X_train_grouped,
    y_train_grouped
)

grouped_scores = grouped_model.predict_proba(
    X_test_grouped
)[:, 1]

grouped_p50 = precision_at_k(
    grouped_scores,
    y_test_grouped,
    k=50
)

# ---------------------------------------------------------
# 5. Report both numbers
# ---------------------------------------------------------

print("VALIDATION AUDIT")
print("=" * 50)

print("\nBEFORE — Row-random split")
print("Train rows:", len(X_train_random))
print("Test rows:", len(X_test_random))
print("Test base rate:", f"{y_test_random.mean():.3f}")
print("Precision@50:", f"{random_p50:.3f}")

print("\nAFTER — Client-grouped split")
print("Train rows:", len(X_train_grouped))
print("Test rows:", len(X_test_grouped))
print("Train clients:", df.iloc[train_idx]["client_id"].nunique())
print("Test clients:", df.iloc[test_idx]["client_id"].nunique())

client_overlap = set(
    df.iloc[train_idx]["client_id"]
).intersection(
    set(df.iloc[test_idx]["client_id"])
)

print("Client overlap:", client_overlap)
print("Test base rate:", f"{y_test_grouped.mean():.3f}")
print("Precision@50:", f"{grouped_p50:.3f}")

print("\nCOMPARISON")
print("=" * 50)

comparison = pd.DataFrame({
    "Validation": [
        "Row-random",
        "Client-grouped"
    ],
    "Precision@50": [
        random_p50,
        grouped_p50
    ],
    "Test base rate": [
        y_test_random.mean(),
        y_test_grouped.mean()
    ]
})

print(comparison.to_string(index=False))

print("\nPrecision@50 change:",
      f"{grouped_p50 - random_p50:+.3f}")

assert len(client_overlap) == 0
print("\nGrouped split check passed: no client appears in both sets.")

VALIDATION AUDIT

BEFORE — Row-random split
Train rows: 24000
Test rows: 6000
Test base rate: 0.542
Precision@50: 0.780

AFTER — Client-grouped split
Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Client overlap: set()
Test base rate: 0.511
Precision@50: 0.380

COMPARISON
    Validation  Precision@50  Test base rate
    Row-random          0.78        0.542000
Client-grouped          0.38        0.510952

Precision@50 change: -0.400

Grouped split check passed: no client appears in both sets.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## Answer

I repeated the leakage hunt on the final Week-5 feature set.

The final model features contain neither `trend_direction` nor `trend_pct`, and they also exclude the recent performance windows used to construct the trend outcome, overlapping 90-day performance fields, identifiers, and existing-system decision metadata.

I also repeated the deliberate leakage test using `trend_pct` alone. Because `trend_pct` is directly connected to the definition of the decline label, a near-perfect result is expected. This is a useful negative control: it shows that the audit can detect a feature that effectively contains the answer.

The important result is therefore not the high leaky score. It is that the leaky fields are absent from the final feature set while the deliberate test still detects the leakage when it is intentionally introduced.


In [3]:
# Section 3 — final feature-set leakage audit

import pandas as pd

# Reuse the feature definitions from Section 2
forbidden_features = [
    # Direct label-derived fields
    "trend_direction",
    "trend_pct",

    # Direct trend-window inputs
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",

    # Overlapping 90-day performance fields
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",

    # Existing-system/search-performance metadata
    "impression_tier",
    "position_tier",
    "provider_used",
    "model_used",

    # Identifiers
    "content_id",
    "client_id",
]

feature_source_fields = numeric_features + categorical_features

leakage_present = [
    col
    for col in forbidden_features
    if col in feature_source_fields
]

print("Final feature count:", len(feature_source_fields))
print("Forbidden columns found in final feature set:")
print(leakage_present)

assert len(leakage_present) == 0, (
    "Leakage detected in the final feature set!"
)

print("\nFinal feature-set check passed.")

# ---------------------------------------------------------
# Deliberate leakage test
# ---------------------------------------------------------

y = (df["trend_direction"] == "down").astype(int)

leaky_prediction = (
    pd.to_numeric(
        df["trend_pct"],
        errors="coerce"
    )
    .fillna(0)
    < -20
).astype(int)

leakage_accuracy = (
    leaky_prediction == y
).mean()

print("\nDeliberate leakage test")
print("-" * 40)
print(
    "Accuracy using trend_pct alone:",
    f"{leakage_accuracy:.3f}"
)

print(
    "\nThe high score is expected because trend_pct is tied "
    "directly to the decline-label definition."
)

# Strong sanity check based on the Week-3 audit
assert leakage_accuracy >= 0.99

print("\nLeakage attack passed.")

Final feature count: 14
Forbidden columns found in final feature set:
[]

Final feature-set check passed.

Deliberate leakage test
----------------------------------------
Accuracy using trend_pct alone: 1.000

The high score is expected because trend_pct is tied directly to the decline-label definition.

Leakage attack passed.


[link text](https://)## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Answer

### Claim rewrite

**The model predicts which pages will decline and can identify the pages that should be refreshed first.**

**Safe version:**

> In this evaluation, the Logistic Regression model produced a measured Precision@50 of 0.380 on the client-grouped holdout, against a 0.511 decline-proxy base rate. This is directional decision-support evidence from the evaluated dataset, not proof that the model can predict future page decline or that refreshing a flagged page will improve performance. A future time-aware evaluation would be needed to test that stronger claim.

This wording keeps the claim tied to what was actually measured: a client-held-out ranking result on a dataset-defined decline proxy.



In [4]:
# Section 4 — print the measured quantities used in the safe claim

print("SAFE CLAIM CHECK")
print("=" * 50)

print(
    "Client-grouped Precision@50:",
    f"{grouped_p50:.3f}"
)

print(
    "Client-grouped test base rate:",
    f"{y_test_grouped.mean():.3f}"
)

print(
    "\nInterpretation:"
)

print(
    "The metric describes the observed evaluation result on the "
    "client-grouped holdout."
)

print(
    "It does not establish a causal effect of refreshing content."
)

print(
    "A future time-aware evaluation would be needed to test "
    "future-decline prediction more directly."
)

SAFE CLAIM CHECK
Client-grouped Precision@50: 0.380
Client-grouped test base rate: 0.511

Interpretation:
The metric describes the observed evaluation result on the client-grouped holdout.
It does not establish a causal effect of refreshing content.
A future time-aware evaluation would be needed to test future-decline prediction more directly.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.